# MoE load-balancing lab

Why do Mixture-of-Experts routers pile tokens onto a few experts, and which fixes produce a model that actually uses
all of its experts? All CPU-scale results are already in `results/` (see `README.md`). This notebook is for the parts
that need a GPU or Hugging Face access, and for re-running anything.

| experiment | what | where it runs |
|---|---|---|
| E1 / E2 | mechanisms and fixes (synthetic) | CPU, results in repo |
| E4 / E5 / E6 | per-method tuning, pre-registered held-out test, depth & representation pathway | CPU, results in repo |
| E3 v2 `--preset cpu` | 4-layer byte-level MoE LM | CPU, results in repo |
| **E3 v2 `--preset gpu`** | 6-layer, 32-expert LM, 10k steps | **GPU (this notebook)** |
| **E7** | routing over a real MoE's pretraining checkpoints (OLMoE) | **GPU + Hugging Face (this notebook)** |

In [ ]:
# Clone the repo. For a private repo, use a token: https://<TOKEN>@github.com/westhauxstripclub/colab.git
!git clone -b claude/moe-load-balancing-overfitting-grpouv https://github.com/westhauxstripclub/colab.git moe-lab
%cd moe-lab
!pip -q install -r requirements.txt
!python -m pytest -q tests

## Quick look: one run without balancing vs. one with a strong aux loss

In [ ]:
import sys; sys.path.insert(0, '.')
import matplotlib.pyplot as plt
from moe_lab.train_synthetic import SynthRunConfig, run_synthetic

base = dict(moe=dict(d_hidden=8, n_experts=32), task=dict(n_clusters=32, n_train=32000, n_val=8000), seed=0)
runs = {name: run_synthetic(SynthRunConfig(name=name, **{**base, 'moe': {**base['moe'], **extra}}))
        for name, extra in [('none', {}), ('aux_1e-1', dict(aux_loss_coef=0.1))]}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (name, r) in zip(axes, runs.items()):
    f = r['final']
    ax.bar(range(len(f['load'])), sorted(f['load'], reverse=True))
    ax.set_title(f"{name}: test loss {f['test_loss']:.3f}, dead experts {f['dead_experts']}")
    ax.set_xlabel('expert (sorted by load)'); ax.set_ylabel('share of tokens')
plt.show()

## Scaled language model on a GPU (E3 v2, `gpu` preset)

6 layers, d_model 256, 32 experts × 128 (top-2), context 256, 10k steps, 3 seeds, 13 methods. Rough cost on an
A100: ~10–15 min per run; use `--methods` to run a subset first (e.g. `none aux_1e-2 aux_1e-2_then_release
frozen_router detach_input dense_active dense_total`). Results are cached per run, so re-running resumes.
This preset has only been smoke-tested (a few steps on CPU), not run end to end.

In [ ]:
%cd experiments
!python e3_lm.py --preset gpu --device cuda --workers 1 --threads 2 --methods none aux_1e-2 aux_1e-2_then_release frozen_router detach_input dense_active dense_total

## When does routing settle in a real MoE? (E7, OLMoE checkpoints)

Needs Hugging Face access and a GPU with ≥ 24 GB (OLMoE-1B-7B is ~14 GB in bf16). Tested in the repo only on tiny
randomly initialised OLMoE-architecture models. OLMoE was trained *with* load balancing, so this shows when routing
settles under balancing, not unbalanced collapse.

In [ ]:
!pip -q install -r ../requirements-colab.txt
!python e7_real_checkpoints.py --repo allenai/OLMoE-1B-7B-0924 --max_checkpoints 10 --device cuda

## Re-run the CPU experiments (optional; results are already in the repo)

Colab CPUs are slow; `--seeds 4` gives a quick pass. E5 must only be run after E4 `select` (see `PREREGISTRATION.md`).

In [ ]:
# !python e1_mechanisms.py --seeds 20 --workers 2
# !python e4_tuning.py tune && python e4_tuning.py select && python e4_tuning.py eval
# !python e5_heldout.py && python e6_deep.py
# !python e3_lm.py --preset cpu --workers 2

In [ ]:
!python analyze.py && python analyze2.py
from IPython.display import Image, Markdown, display
import glob
for name in sorted(glob.glob('../results/summary_*.md')):
    display(Markdown(open(name).read()))
for p in sorted(glob.glob('../results/figures/*.png')):
    display(Image(p))